# Qwen3-14B QLoRA: reasoning behaviour and transfer

This project originated from an **MMA 606 course assignment**. The training notebook was adapted from the **course-provided starter workflow**, itself attributed to the [Hugging Face TRL SFT / LoRA / QLoRA example](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/notebooks/sft_trl_lora_qlora.ipynb).

This cleaned notebook preserves the submitted notebook's active **Code-Feedback** training configuration and classical baseline. The companion report also describes an earlier ORCA Math adapter and qualitative transfer probes; this notebook is not a complete executable record of both experiments. See [README](../README.md) and [experiment notes](../docs/experiment_notes.md) for the evidence, reported findings, and limitations.

Saved outputs and execution counts were cleared for publication. The original confusion-matrix image is retained separately in [assets](../assets/baseline_confusion_matrix.png). No training or evaluation was rerun during repository preparation.

## Environment and execution

Install `requirements.txt` in a compatible GPU environment. The setup cell below accepts a notebook launched from either the repository root or `notebooks/`. The original run used an NVIDIA A100-SXM4-40GB GPU. A CUDA GPU, compatible bitsandbytes/Liger installation, and bfloat16 support are required for the unchanged training setup; the original run does not establish a smaller-GPU minimum. Package versions were not recorded as a complete lockfile.

Run the notebook in order from a fresh kernel. Public model and dataset files are downloaded from Hugging Face. Training artifacts are saved locally; automatic Hub uploads and Trackio publishing have been removed. No account login or embedded credential is needed for the public resources used here. The optional merge/upload/vLLM serving appendix from the starter workflow was omitted; it is not part of the reported experiment.

In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.is_file():
    requirements_path = Path("../requirements.txt")
if not requirements_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")

%pip install -r {requirements_path}

## Load the Code-Feedback training split

The active submission uses `HuggingFaceH4/Code-Feedback`, split `train_sft`. Its saved dataset summary contains 65,383 rows with `id` and `messages` fields. Training below stops after 88 optimizer steps; loading the full split does not mean the model completed an epoch.

In [ ]:
from datasets import load_dataset

dataset_name = "HuggingFaceH4/Code-Feedback"
train_dataset = load_dataset(dataset_name, split="train_sft")

In [ ]:
train_dataset

Inspect one training example and remove its identifier before supervised fine-tuning.

In [ ]:
train_dataset[0]

In [ ]:
train_dataset = train_dataset.remove_columns(column_names=['id'])

## Load the quantized base model

QLoRA trains low-rank adapters while the base model remains quantized. The loading code requests 4-bit NF4 quantization, double quantization, bfloat16 computation, and PyTorch scaled-dot-product attention (`sdpa`) for Qwen3-14B. The source output warns that this prequantized checkpoint supplies its own quantization configuration, which takes precedence; these are the requested settings.

In [ ]:
model_id, output_dir = "unsloth/qwen3-14b-unsloth-bnb-4bit", "qwen3-14b-unsloth-bnb-4bit-SFT2"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    use_cache=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)

The adapter targets attention and feed-forward projections with rank 32 and alpha 32.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## Train the adapter

The submitted settings are preserved. With one GPU, batch size 1 and four accumulation steps give an effective batch size of **4**. The original comment incorrectly described 16. No evaluation dataset was supplied, so the saved training loss is not a held-out evaluation result.

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=88,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=1,
    report_to="none",  # Keep experiment logging local for public reuse.
    output_dir=output_dir,
    max_length=2048,
    use_liger_kernel=True,
    activation_offloading=False,
    push_to_hub=False,
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)

Measure reserved GPU memory before training.

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

Record elapsed time and peak reserved GPU memory. These memory values measure PyTorch reservations, including the loaded model, and are specific to this runtime. The difference from the pre-training reservation is not a general hardware requirement.

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Save the adapter locally

In [ ]:
trainer.save_model(output_dir)

## Compare base and adapter responses

Reload the same quantized base model and use the locally saved adapter. This preserves the original loading and generation settings. Reloading creates a second model allocation while the trainer remains in memory, so this section can require substantial GPU memory.

The current prompt is the submitted Python mutable-default-argument probe. The report contains other hand-selected probes, but this notebook does not retain their complete outputs. Both generation calls retain a limit of 512 new tokens and leave other decoding settings at their model/library defaults. No generation seed is set.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_model = output_dir

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    use_cache=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
messages = [
    {
        'content': "What will the following Python code print? Explain why.\n\ndef append_to(element, to=[]):\n    to.append(element)\n    return to\n\nprint(append_to(1))\nprint(append_to(2))\nprint(append_to(3))",
        'role': 'user',
    }
]

### Base response

Attaching a PEFT adapter modifies the shared model object. The guard below disables the adapter if this cell is rerun after attachment, so the base response still uses base weights. This is a publication-time execution safeguard, not a rerun or correction of the historical outputs. The original notebook's out-of-order execution counts make its saved base-versus-adapter comparison ambiguous.

In [ ]:
from contextlib import nullcontext

text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

adapter_context = (
    fine_tuned_model.disable_adapter()
    if "fine_tuned_model" in globals() and fine_tuned_model.get_base_model() is base_model
    else nullcontext()
)
with adapter_context:
    generated_ids = base_model.generate(
        **model_inputs,
        max_new_tokens=512,
    )
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

### Adapter response

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

## Classical question-answer matching baseline

This is a separate binary classification task, not a directly comparable LLM reasoning benchmark. From the first 2,000 ORCA Math examples, create 2,000 original question-answer pairs (label 1) and 2,000 pairs with a randomly selected different answer (label 0). Concatenate each pair with `[SEP]`, encode with up to 5,000 TF-IDF features, and fit logistic regression.

The original procedure is deliberately preserved, including its limitations: TF-IDF is fitted **before** the split; questions and answers can recur across the training and test sets; the stratified 75/25 split has no fixed random state; and an index-mismatched answer is not independently verified as semantically incorrect. The negative-pair generator alone uses seed 10086. These choices prevent a clean, reproducible estimate of semantic generalization. The five-fold dummy-classifier calculation is retained but the original notebook did not print its scores.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
from datasets import load_dataset
import random
import warnings
warnings.filterwarnings('ignore')

In [ ]:
dataset = load_dataset("HuggingFaceH4/orca-math-word-problems-200k", split="train_sft[:2000]")

In [ ]:
questions = dataset['question']
answers = dataset['answer']
texts = []
labels = []

In [ ]:
for i in range(len(questions)):
    text = questions[i] + " [SEP] " + answers[i]
    texts.append(text)
    labels.append(1)

In [ ]:
random.seed(10086)

In [ ]:
for i in range(len(questions)):
    # Pick a random different answer
    wrong_idx = (i + random.randint(1, len(answers) - 1)) % len(answers)
    text = questions[i] + " [SEP] " + answers[wrong_idx]
    texts.append(text)
    labels.append(0)

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
features = vectorizer.fit_transform(texts)

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(features, labels, test_size=0.25, stratify=labels)

In [ ]:
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_scores = cross_val_score(dummy_clf, train_features, train_labels, cv=5)

In [ ]:
lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(train_features, train_labels)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

Display the logistic-regression confusion matrix and accuracy for this run. The original saved figure is retained in `assets/baseline_confusion_matrix.png`; a fresh unseeded split can produce different results.

In [ ]:
# Predict
predictions = lr_clf.predict(test_features)
# Confusion Matrix (one visual output)
cm = confusion_matrix(test_labels, predictions)
disp = ConfusionMatrixDisplay(cm, display_labels=['Wrong Answer', 'Correct Answer'])
disp.plot(cmap='Blues')
plt.title(f'Logistic Regression — Accuracy: {accuracy_score(test_labels, predictions):.1%}')
plt.show()

## Interpretation

Treat the report's transfer examples as exploratory qualitative observations. The retained notebook does not provide a systematic benchmark, repeated-seed study, or complete response archive for both adapters. The original saved base and adapter outputs for the current mutable-default probe both echo the prompt code rather than answering it; they do not independently substantiate the report's earlier comparisons. Consult [experiment notes](../docs/experiment_notes.md) before interpreting the report's broader conclusions.